## 第15章 自省和泛型

### 1.特殊属性

- **自省**：自省是代码在运行时访问有关自身的信息并相应做出响应的能力。
- **特殊属性**：Python主要通过将重要信息存储在所使用的不同对象的特殊属性中来实现自省。和特殊方法一样，所有特殊属性都以双下画线`__`开头和结尾。
    - 模块的特殊属性：

        <p><img src="img/007.jpg" width="650"></p>

    - 函数的特殊属性：
        
        <p><img src="img/008.jpg" width="650"></p>
    
    - 对象的特殊属性：
        
        <p><img src="img/009.jpg" width="650"></p>

    - 类的特殊属性：
        
        <p><img src="img/010.jpg" width="650"></p>

### 2.查看对象(类)内部

- **`__dict__`**：存储类或对象所有属性和方法的字典。
    - 注意：类的`__dict__`属性是一种MappingProxyType(只读映射代理)，只有`type.__setattr__()`能修改。对象的`__dict__`属性是一个普通字典，可以直接修改。
    - 注意：实例(对象)属性存储在实例`__dict__`属性中，实例方法以及类属性、类方法存储在类`__dict__`属性中。

In [ ]:
class Quadruped:
    leg_count = 4

    def __init__(self, species):
        self.species = species

class Llama(Quadruped):
    """A quadruped that lives in large rivers."""
    dangerous = True

    def __init__(self):
        super().__init__("llama")
        self.swimming = False

    def warn(self):
        if self.swimming:
            print("Cuidado, llamas!")

    @classmethod
    def feed(cls):
        print("Eats honey with beak.")

from pprint import pprint

llama = Llama()
pprint(llama.__dict__)  # 输出对象的__dict__属性
# {'species': 'llama', 'swimming': False}

pprint(Llama.__dict__)  # 输出类的__dict__属性
# mappingproxy({'__doc__': 'A quadruped that lives in large rivers.',
#               '__init__': <function Llama.__init__ at 0x000001C224BC7920>,
#               '__module__': '__main__',
#               'dangerous': True,
#               'feed': <classmethod(<function Llama.feed at 0x000001C224BC7B00>)>,
#               'warn': <function Llama.warn at 0x000001C224BC6D40>})

pprint(Quadruped.__dict__)  # 输出父类的__dict__属性
# mappingproxy({'__dict__': <attribute '__dict__' of 'Quadruped' objects>,
#               '__doc__': None,
#               '__init__': <function Quadruped.__init__ at 0x000001C224BC6DE0>,
#               '__module__': '__main__',
#               '__weakref__': <attribute '__weakref__' of 'Quadruped' objects>,
#               'leg_count': 4})

- **列出属性**：
    - `var(obj)`函数返回指定对象或类的`__dict__`属性。
    - `dir(obj)`函数返回指定对象或类所有属性(包含特殊属性、自定义属性、继承的属性)的名称(不包含值)的列表。
    - 在实践中，vars()、dir()函数通常仅在交互式提示环境中工作，或仅在调试期间有用。

In [ ]:
# 与上面输出一致
pprint(vars(llama))
pprint(vars(Llama))
pprint(vars(Quadruped))

# 输出llama的所有属性
pprint(dir(llama))

- **获取属性**：
    - 点`.`运算符：用于获取对象的属性值。本质上是内置函数`getattr()`的一种简化写法。
    - `getattr(obj, attr)`：返回对象`obj`的属性`attr`的值。实际上调用了两个特殊方法：`__getattribute__()`用于处理复杂的查找逻辑；用户选择性实现的`__getattr__()`。
        - 通常只应覆写 `__getattr__()`，**不要碰 `__getattribute__()`**。
        - `__getattr__()`常见用途：为不存在的属性提供默认值。
    - 内部执行逻辑：

        ```python
        getattr(obj, name)
        → 调用 __getattribute__(obj, name)   # 默认由 object/type 提供
            → 沿 MRO 搜索 __dict__
            → 找到 → 返回值
            → 未找到 → 抛出 AttributeError
            → 检查是否定义了 __getattr__()
                → 有 → 调用 __getattr__() 作为兜底
                → 无 → 重新抛出 AttributeError
        ```

In [ ]:
print(llama.swimming)
print(getattr(llama, 'swimming'))
print(object.__getattribute__(llama, 'swimming'))  # 对象 -> 底层调用object.__getattribute__()

print(Llama.dangerous)
print(getattr(Llama, 'dangerous'))
print(type.__getattribute__(Llama, 'dangerous'))   # 类 -> 底层调用type.__getattribute__()

- **检查属性**：`hasattr(obj, attr)`，本质上hasattr()是在try语句中调用了getattr()。
    - `try: getattr(obj, attr) except AttributeError: return False`。
- **设置属性**：`setattr(obj, attr, value)`，依赖__setattr__()特殊方法。
    - `object.__setattr__(obj, attr, value)`。
    - 它只关心指定对象或类的__dict__，如果该属性存在于__dict__中就修改它，否则就在__dict__中创建一个新属性。
        - 这样容易导致属性意外覆盖。因此，**修改类属性时，必须在类上操作，而非实例上。**
- **删除属性**：`del obj.atr` 或 `delattr(obj, attr)`，依赖于__delattr__()特殊方法。

### 3.描述符

- **描述符**：描述符是具有绑定行为的对象属性。**它是一个实现了描述符协议方法的类，仅在用作另一个类的类属性时才会表现出绑定行为，从而接管对该属性的读写删除操作。**
    - 描述符协议方法：
        - `__get__(self, instance, owner=None)`：读取属性时调用，返回属性的值。
        - `__set__(self, instance, value)`：设置属性时调用，将值赋值给属性。
        - `__delete__(self, instance)`：删除属性时调用，删除属性。
        - `__set_name__(self, owner, name)`：该方法不是描述符协议方法，当绑定描述符到一个名称时该方法将被调用。
    - 描述符分类：
        - 数据描述符：实现了`__get__`、`__set__`和/或`__delete__`方法。
        - 非数据描述符：仅实现了`__get__`方法。
        - 注意：**Python搜索属性时，数据描述符具有最高优先级，其次是存储在对象的__dict__中的普通属性，然后是非数据描述符，最后是类及其基类中的任何属性。**
            - 内置描述符：@property、@classmethod、@staticmethod、super()等。
    - 描述符原理：当通过实例访问一个属性时，Python遵循以下查找顺序：
        - 首先在实例字典(`obj.__dict__`)中查找。
        - 然后在类字典(`type(obj).__dict__`)中查找。
        - 继续在父类字典(`父类.__dict__)`中查找。
        - 如果在类字典(或父类字典)中找到的值是一个描述符:
            - 如果是数据描述符，直接调用描述符方法，跳过实例字典.
            - 如果是非数据描述符，先检查实例字典，如果实例字典中有同名属性则使用实例字典的值，否则调用描述符的`__get__`方法。
    - 描述符作用：
        - 属性验证与类型检查：描述符可以将验证逻辑从类中剥离出来，实现代码复用。
        - 惰性求值：利用"非数据描述符优先级低于实例字典"的特性，可以实现"首次访问时计算、之后直接返回缓存值"的惰性属性。
        - 只读属性：利用"数据描述符优先级高于实例字典"的特性，可以实现只读属性。
        - 动态计算属性：描述符可以在每次访问时动态计算值。


In [ ]:
import re

# 描述符类：实现描述符协议
class Book:
    pattern = re.compile(r'(.+)\((\d+)\)\. (.+)\. (.+)\..*')

    # 绑定到属性时自动调用，保存描述符实例的属性名
    def __set_name__(self, owner, name):
        self.name = name

    # 根据描述符实例的属性名和指定的属性名生成属性名，防止一个实例中多次使用同个描述符出现冲突
    def attr(self, attr):
        return f'{self.name}.{attr}'

    # 设置属性时自动调用
    def __set__(self, instance, value):
        matches = self.pattern.match(value)
        if not matches:
            raise ValueError("Book data must be specified in APA 7 format.")

        # 设置属性值，绑定到实例属性，属性名通过attr方法生成
        setattr(instance, self.attr('author'), matches.group(1))
        setattr(instance, self.attr('year'), matches.group(2))
        setattr(instance, self.attr('title'), matches.group(3))
        setattr(instance, self.attr('publisher'), matches.group(4))

    # 获取属性时自动调用
    def __get__(self, instance, owner=None):
        try:
            title = getattr(instance, self.attr('title'))
            author = getattr(instance, self.attr('author'))
        except AttributeError:
            return 'Nothing right now.'
        return f'{title} by {author}'

    # 删除属性时自动调用
    def __delete__(self, instance):
        delattr(instance, self.attr('author'))
        delattr(instance, self.attr('year'))
        delattr(instance, self.attr('title'))
        delattr(instance, self.attr('publisher'))


class BookClub:
    reading = Book()        # 将描述符绑定到类属性reading
    reading_next = Book()   # 将描述符绑定到类属性reading_next

    def __init__(self, name):
        self.name = name
        self.members = []

    def new_member(self, member):
        self.members.append(member)
        print(
            "===== - - - - - - - - - =====",
            f"Welcome to the {self.name} Book Club, {member}!",
            f"We are reading {self.reading}",
            "===== - - - - - - - - - =====",
            sep='\n'
        )

# 创建实例
mystery_lovers = BookClub("Mystery Lovers")

# 设置属性
mystery_lovers.reading = "McDonald, J. C. (2019). Noah Clue, P.I. AJ Charleson Publishing."
mystery_lovers.reading_next = "Chesterton, G.K. (1911). The Innocence of Father Brown. Cassell and Company, Ltd."

print(f"Now: {mystery_lovers.reading}")
print(f"Next: {mystery_lovers.reading_next}")

import pprint
pprint.pprint(dir(mystery_lovers))
# [ ...
#  'members',
#  'name',
#  'new_member',
#  'reading',
#  'reading.author',
#  'reading.publisher',
#  'reading.title',
#  'reading.year',
#  'reading_next',
#  'reading_next.author',
#  'reading_next.publisher',
#  'reading_next.title',
#  'reading_next.year']

### 4.slots

- **slots**：在Python类中定义一个类变量`__slots__`，显式声明该类的实例允许拥有的属性名称集合，并且其实例不再自动创建`__dict__`字典。
    - 定义规则：`__slots__ = ('attr1', 'attr2', ...)`。
        - `__slots__` 是一个**属性名的元组**。
        - 只包含**实例属性**名，不含方法名和类属性名(它们存在类的 `__dict__` 中)。
        - **slot 名不得与类属性名冲突**(`__dict__` 和 `__weakref__` 除外)。
    - 主要作用：
        - 减少内存占用：默认情况下每个Python实例都有一个`__dict__`字典，非常消耗内存。`__slots__`使用类似数组的方式存储属性，能极大地减少了内存占用。
        - 提升属性访问速度：由于去掉了字典查找这一层间接性，属性访问变成了更快的数组索引或指针偏移操作。
        - 防止动态添加属性：在类定义时就确定了允许的属性名称，后续不能动态添加或删除属性。
    - 应用场景：
        - 需要创建海量实例(百万级以上)，内存敏感的应用。
        - 希望防止外部随意添加属性，保持数据结构的规范性。
    - 注意事项：
        - 如果想在运行时动态添加或删除属性，可以将`__dict__`添加到`__slots__`中。
        - 对应继承，首先，应该只在继承树中声明任何给定的slots一次；其次，不能从多个具有非空slots的父类继承。

In [ ]:
import sys

class WithoutSlots:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class WithSlots:
    __slots__ = ('x', 'y')    # 声明该类的实例只有 x 和 y 两个属性
    def __init__(self, x, y):
        self.x = x
        self.y = y

w1 = WithoutSlots(1, 2)
w2 = WithSlots(1, 2)

# w2.z = 3  # 报错：不能动态添加属性

print(sys.getsizeof(w1.__dict__))  # 296字节
print(sys.getsizeof(w2))           # 48字节，节约内存

### 5.不可变类

- **不可变类**：不可变类是指实例创建后，其内部状态(属性值)再也无法被修改的类。比如：str、int、tuple等。
    - 实现方式：
        - `@dataclass(frozen=True)`：现代Python中最简洁、最标准的方式。
        - `__slots__` + `__setattr__` + `__delattr__`：最传统的手动控制方式，可以精确控制"哪些属性可以改、哪些不能"。
            - `__setattr__()`方法中，必须调用`object.__setattr__()`，不能调用`setattr()`函数，否则无限递归。
    - 注意事项：**绝大多数实现只做到"浅层不可变"，如果属性指向一个可变对象，你无法替换这个属性，但可以修改其内部内容。**

In [ ]:
class ImmutablePoint:

    __slots__ = ('x', 'y')      # 限制属性，避免动态添加属性

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __setattr__(self, name, value):  # 重写setattr方法，限制属性修改，除了第一次赋值
        if hasattr(self, name):
            raise AttributeError(f"{type(self)} object attribute '{name}' is immutable")
        object.__setattr__(self, name, value)

    def __delattr__(self, name):  # 重写delattr方法，限制属性删除
        raise AttributeError(f"{type(self)} object attribute '{name}' is immutable")

p = ImmutablePoint(1, 2)
p.x = 3     # 报错

### 6.单分派泛型函数

- **单分派泛型函数**：是一种根据第一个参数的类型来决定调用哪个具体实现的函数。
    - 简单说明：同一个函数名，通过"注册机制"将类型与实现绑定，传入不同类型的参数时，会自动执行不同的代码逻辑，从而实现类似其他语言中"函数重载"的效果。
    - 实现方式：先通过`@singledispatchmethod`(类方法)或`@singledispatch`(函数)装饰器定义一个默认实现，然后通过`register`方法注册其他实现。
        - 用类型提示注册：通过第一个参数的类型提示进行注释。
        - 用显示类型注册：在`register`方法中指定具体的类型参数。
        - 用`register`方法注册：直接在`register`方法中指定类型并结合lambda表达式进行注册。
    - 关键要点：
        - 默认实现方法必须**最先**声明，用 `@singledispatchmethod` 装饰器。
        - `@singledispatchmethod` 必须是**最外层**装饰器。

In [ ]:
from functools import singledispatchmethod

class Element:
    def __init__(self, symbol):
        self.symbol = symbol

    # 先定义默认实现
    @singledispatchmethod
    def __eq__(self, other):
        return self.symbol == other.symbol

    # 通过类型提示注册
    @__eq__.register
    def _(self, other: str):
        return self.symbol == other

    # 通过显示类型注册
    @__eq__.register(int)
    def _(self, other):
        return self.symbol == other

    # 通过register方法注册
    __eq__.register(float, lambda self, other: self.symbol == other)

# 类似其他语言的函数重载
e = Element('H')
print(e == 'h')
print(e == 1)
print(e == 1.0)

### 7.任意执行

- `eval()`函数：将字符串转换为Python表达式并执行。
- `exec()`函数：将字符串转换为Python语句并执行。
- `compile()`函数：将字符串转换为Python代码对象。

In [ ]:
nums = [2, 4, 6, 8]

for num in nums:
    expression = f"{num} // 2 + 2"
    try:
        answer = eval(expression)  # 执行表达式
    except (NameError, ValueError, TypeError, SyntaxError) as e:
        print(e)
    finally:
        code = "print('The answer is', answer)"
        obj = compile(code, '<string>', mode='exec')  # 编译成代码对象
        exec(obj)  # 执行代码对象

### 8.本章小结

- **核心知识脉络**

```text
内省与泛型
│
├── 一、内省基础 —— 查看对象的内部
│   ├── vars() / locals() / globals() / dir()
│   ├── getattr() → __getattribute__() → __getattr__()
│   ├── hasattr()
│   ├── setattr() → __setattr__()
│   └── delattr() → __delattr__()
│
├── 二、函数属性
│   ├── 函数也是对象 → 可以有属性
│   ├── 函数属性 ≠ 局部变量（存在 __dict__ 中）
│   ├── 陷阱：可变性导致状态共享 → 逻辑炸弹
│   └── 正确用法：装饰器提供默认值，运行时不变
│
├── 三、描述符⭐⭐
│   ├── 描述符协议：__get__ / __set__ / __delete__
│   │   ├── 非数据描述符（仅 __get__）→ 如方法
│   │   └── 数据描述符（含 __set__ 或 __delete__）→ 如 property
│   ├── 查找链优先级：数据描述符 > 实例 __dict__ > 非数据描述符
│   ├── 错误写法：数据保存在描述符自身 → 多实例共享同一数据
│   ├── 正确写法：数据保存在 instance 上 → 每个实例独立
│   ├── 多描述符冲突：同一类中多个同名描述符覆盖属性
│   │   └── 解决：__set_name__() + 命名空间前缀（如 reading.title）
│   └── 描述符必须是类属性！实例属性无描述符行为
│
├── 四、Slots⭐⭐
│   ├── 作用：预声明实例属性 → 提升性能、降低内存
│   ├── 声明方式：__slots__ = ('attr1', 'attr2', ...)
│   ├── 限制：只能使用已声明的属性名
│   ├── 灵活折中：加入 '__dict__' 恢复动态属性
│   ├── '__weakref__'：支持弱引用
│   └── 继承规则：
│       ├── 同一 slot 不可在继承树中重复声明
│       └── 多继承时，父类应为空 __slots__
│
├── 五、不可变类⭐⭐
│   ├── Python 无真正的不可变类机制 → 需手动模拟
│   ├── 实现三件套：
│   │   ├── __slots__（不含 '__dict__'）→ 禁止新增属性
│   │   ├── __setattr__() → 属性已存在则拒绝修改
│   │   └── __delattr__() → 拒绝删除
│   └── 不可变对象 = 可哈希 → 可作字典键
│
├── 六、单分派泛型函数⭐⭐
│   ├── @singledispatch（函数）/ @singledispatchmethod（方法）
│   ├── 根据第一个参数的类型分发到不同实现
│   ├── 三种注册方式：
│   │   ├── 类型注解 + @__eq__.register
│   │   ├── 显式类型 + @__lt__.register(str)
│   │   └── register() 方法 + lambda
│   ├── @typing.overload → 多类型共享同一实现
│   └── 惯例：分派方法名用下划线 _
│
└── 七、任意执行（Arbitrary Execution）⚠️
    ├── eval() / compile() / exec()
    ├── 致命风险：代码注入攻击
    ├── 安全替代：ast.literal_eval()（但不含运算符）
    └── 铁律：永远不要对外部数据使用 eval/exec
```

- **警告与提示表**

| 类型     | 内容                                                         |
| -------- | ------------------------------------------------------------ |
| ⚠️ 陷阱   | 函数属性 ≠ 局部变量，函数属性存储在函数的 `__dict__` 中      |
| ⚠️ 陷阱   | 描述符数据存在 `self` 上 → 多实例共享，数据互相覆盖          |
| ⚠️ 规则   | 描述符**必须是类属性**，实例属性上无描述符行为               |
| ⚠️ 限制   | `__slots__` 中 slot 名不能与同类属性名冲突（`__dict__`/`__weakref__` 例外） |
| ⚠️ 继承   | 多继承时父类不能都有非空 `__slots__`                         |
| ⚠️ 安全   | `eval()` / `exec()` 对外部数据 = 代码注入攻击 = 灾难         |
| 💡 提示   | 不可变类中的 `__setattr__()` 必须用 `object.__setattr__()` 避免递归 |
| 💡 提示   | `@singledispatchmethod` 必须是最外层装饰器                   |
| 💡 提示   | 单分派方法惯例用 `_` 作名，避免命名冲突                      |
| 💡 提示   | `__getattribute__()` 通常不应覆写，只覆写 `__getattr__()`    |